# ShopSense 07: Fix the answer step and measure again

**Goal:** fix the problems notebook 06 found, then show **before and after** numbers on the same questions.

**What notebook 06 found, and what we do about it**

| Finding | Cause | Fix |
|---|---|---|
| 12 of 24 answers had an uncited sentence | **Half of this was my checker bug:** the model writes `[1, 2, 3]` and the checker only read `[1]`. The other half is real: the model opens with a sentence that has no citation | checker reads both styles; prompt says "write `[1][2]`, never `[1, 2]`"; code **deletes** any sentence that still has no citation |
| 5 answers counted as "not refused" | Checker wanted the exact phrase. The model wrote "The reviewers do not mention the packaging [1]" (a correct refusal in other words) | refusal = first sentence says the reviews do not say/mention/discuss the topic |
| Answers use a review that is about something else (for example, a "fits my dryer" review cited for "value for money") | The small model cites all six excerpts | prompt rule 3: cite an excerpt only if it talks about the topic asked |
| Hybrid search missed the relevant review for G019, G024, G032, G057 | Merging BM25 with vector pushed it out of the top 6 | run the same questions with **vector** search and compare |

**Two parts**
1. **Before:** re-score the old answers (`06_answers.csv`) with the fixed checker. No model call needed.
2. **After:** run the new prompt with vector search on all 39 questions, and with hybrid search on the 22 aspect questions. About 25 minutes in total. Results are saved as they finish, so if Ollama stops you can run the cell again and it continues.

**Reads:** `reports/eval/06_answers.csv`, `data/eval/golden_set.csv`, `data/eval/retrieval_labels.csv`.
**Writes:** `reports/eval/07_answers_vector.csv`, `07_answers_hybrid.csv`, `07_before_after.csv`, `07_hand_read.csv`.

**Needs:** Ollama running with `llama3.2:3b`, and the new `src/shopsense/llm/answer.py`.

### Rule written before running (so we cannot cheat afterwards)
We switch the default search to **vector** only if, in the table below, vector has no more false refusals and no more rule-breaking answers than hybrid. Notebook 06 already showed that vector was not clearly different from hybrid in a statistical sense (the interval included 0), so we will say that honestly either way.

## 1. Setup

**What this does:** imports libraries, loads the search index (about 20 seconds) and the question files.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from shopsense.llm.answer import check_citations, answer_question
from shopsense.retrieval.hybrid import load_index

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

OUT = ROOT / "reports" / "eval"
golden = pd.read_csv(ROOT / "data" / "eval" / "golden_set.csv")
labels = pd.read_csv(ROOT / "data" / "eval" / "retrieval_labels.csv")
no_evidence = labels.groupby("question_id")["relevant"].sum()
no_evidence = no_evidence[no_evidence == 0].index.tolist()

index = load_index()
print("questions with no relevant review (a refusal is fine there):", no_evidence)

d:\Resume\Data Scientist\Resume_Projects\ShopSense_Product Review Intelligence Copilot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6594.31it/s]


questions with no relevant review (a refusal is fine there): ['G003', 'G014', 'G052']


## 2. Before: re-score the old answers with the fixed checker

**What this does:** reads the 39 answers saved by notebook 06 and checks them again with the fixed rules (citation lists like `[1, 2]` are read, and a refusal in other words counts as a refusal). The answers themselves do not change.

In [2]:
old = pd.read_csv(OUT / "06_answers.csv")

rows = []
for row in old.itertuples():
    n_passages = len(str(row.passages).split(" || "))
    cited, invalid, uncited, refused = check_citations(row.answer, n_passages)
    rows.append({"id": row.id, "type": row.type, "refused": refused, "invalid": len(invalid),
                 "broke_citation_rule": uncited > 0, "seconds": row.seconds})

before = pd.DataFrame(rows)
print("old answers re-scored:", len(before))
before[before["id"].isin(["G014", "G019", "G025", "G057", "G065"])]

old answers re-scored: 39


,id,type,refused,invalid,broke_citation_rule,seconds
7,G014,aspect_summary,True,0,False,17.5
10,G019,aspect_summary,True,0,False,16.3
14,G025,unanswerable,True,0,False,27.1
32,G057,aspect_summary,True,0,False,19.9
36,G065,out_of_scope,True,0,False,24.6


## 3. After: run the new prompt

**What this does:** picks the questions. Vector search runs on **all 39** questions. Hybrid search (new prompt) runs on the **22 aspect** questions only, to separate two effects: the new prompt (hybrid before vs hybrid after) and the search method (hybrid after vs vector after).

In [3]:
test_questions = golden[golden["question_type"].isin(["aspect_summary", "unanswerable", "out_of_scope"])].copy()
aspect_questions = test_questions[test_questions["question_type"] == "aspect_summary"]

plan = [("vector", test_questions), ("hybrid", aspect_questions)]
print("calls to the model:", sum(len(q) for _, q in plan), "(about 25 minutes)")

calls to the model: 61 (about 25 minutes)


**What this does:** runs the questions and saves each answer as it finishes. If the file already exists, finished questions are skipped, so you can stop and continue. `raw_answer` is what the model wrote; `answer` is after the code removed uncited sentences; `dropped` counts removed sentences.

In [4]:
results = {}
for method, questions in plan:
    path = OUT / f"07_answers_{method}.csv"
    finished = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["id"])
    rows = finished.to_dict("records")
    done_ids = set(finished["id"])

    for number, q in enumerate(questions.itertuples(), start=1):
        if q.id in done_ids:
            continue
        asin = q.product_asin if isinstance(q.product_asin, str) and q.product_asin != "" else None
        result = answer_question(index, q.question, asin=asin, method=method)
        rows.append({"id": q.id, "type": q.question_type, "question": q.question,
                     "answer": result["answer"], "raw_answer": result["raw_answer"],
                     "dropped": result["dropped_sentences"], "refused": result["refused"],
                     "cited": str(result["cited"]), "invalid": len(result["invalid"]),
                     "seconds": result["seconds"],
                     "passages": " || ".join(result["hits"]["text"].str.slice(0, 400).tolist()) if result["hits"] is not None else ""})
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"{method:6s} {number}/{len(questions)} {q.id} refused={result['refused']} {result['seconds']:.0f}s")

    results[method] = pd.DataFrame(rows)

vector 1/39 G003 refused=False 90s
vector 2/39 G004 refused=False 33s
vector 3/39 G005 refused=True 16s
vector 4/39 G008 refused=False 40s
vector 5/39 G009 refused=False 26s
vector 6/39 G010 refused=True 16s
vector 7/39 G013 refused=False 35s
vector 8/39 G014 refused=True 26s
vector 9/39 G015 refused=True 15s
vector 10/39 G018 refused=False 44s
vector 11/39 G019 refused=False 24s
vector 12/39 G020 refused=True 11s
vector 13/39 G023 refused=False 38s
vector 14/39 G024 refused=False 29s
vector 15/39 G025 refused=True 12s
vector 16/39 G028 refused=True 15s
vector 17/39 G031 refused=False 33s
vector 18/39 G032 refused=False 32s
vector 19/39 G033 refused=True 14s
vector 20/39 G036 refused=False 29s
vector 21/39 G037 refused=False 22s
vector 22/39 G038 refused=True 20s
vector 23/39 G041 refused=False 36s
vector 24/39 G042 refused=False 29s
vector 25/39 G043 refused=True 10s
vector 26/39 G046 refused=False 35s
vector 27/39 G047 refused=False 24s
vector 28/39 G048 refused=True 11s
vector 29/39

## 4. Before and after

**What this does:** one table with the same numbers for three runs.
- **false refusals:** aspect questions that DO have relevant reviews but were refused (should be 0)
- **answered with no evidence:** the 3 aspect questions with no relevant review that were answered instead of refused (lower is better; a refusal there is correct)
- **refusal %:** unanswerable and out-of-scope questions (should be 100)
- **broke citation rule:** answers where the model wrote a sentence without a citation, **before** the code cleaned it (lower is better)
- **invalid citations:** `[n]` that does not exist (should be 0)

The "before" run only has the cleaned numbers it can have: its answers were never cleaned, so its rule-breaking count is what a user would have seen.

In [5]:
runs = {"before: old prompt, hybrid": before}
for method in ["hybrid", "vector"]:
    part = results[method].copy()
    part["broke_citation_rule"] = part["dropped"] > 0
    part["invalid"] = part["invalid"].astype(int)
    runs[f"after: new prompt, {method}"] = part

table = []
for name, part in runs.items():
    aspect = part[part["type"] == "aspect_summary"]
    with_evidence = aspect[~aspect["id"].isin(no_evidence)]
    without_evidence = aspect[aspect["id"].isin(no_evidence)]
    answered = part[~part["refused"]]
    table.append({
        "run": name,
        "false_refusals": int(with_evidence["refused"].sum()),
        "answered_with_no_evidence": int((~without_evidence["refused"]).sum()),
        "unanswerable_refused_%": round(100 * part[part["type"] == "unanswerable"]["refused"].mean(), 1) if (part["type"] == "unanswerable").any() else None,
        "out_of_scope_refused_%": round(100 * part[part["type"] == "out_of_scope"]["refused"].mean(), 1) if (part["type"] == "out_of_scope").any() else None,
        "answers": len(answered),
        "broke_citation_rule": int(answered["broke_citation_rule"].sum()),
        "invalid_citations": int((answered["invalid"] > 0).sum()),
        "avg_seconds": round(part["seconds"].mean(), 1),
    })

before_after = pd.DataFrame(table).set_index("run")
before_after.to_csv(OUT / "07_before_after.csv")
before_after.T

run,"before: old prompt, hybrid","after: new prompt, hybrid","after: new prompt, vector"
false_refusals,2.0,2.0,0.0
answered_with_no_evidence,2.0,2.0,2.0
unanswerable_refused_%,100.0,NaN,100.0
out_of_scope_refused_%,100.0,NaN,100.0
answers,19.0,19.0,21.0
broke_citation_rule,6.0,8.0,11.0
invalid_citations,0.0,0.0,0.0
avg_seconds,27.8,24.0,27.0


**What this does:** shows the answers that changed between hybrid and vector search, so you can see *why* the numbers moved (the new text is what you would show an interviewer).

In [6]:
compare = results["hybrid"][["id", "answer"]].merge(
    results["vector"][["id", "answer"]], on="id", suffixes=("_hybrid", "_vector"))
different = compare[compare["answer_hybrid"] != compare["answer_vector"]]
print("answers that differ:", len(different), "of", len(compare))
for row in different.head(5).itertuples():
    print("\n", row.id)
    print("  hybrid:", row.answer_hybrid[:300])
    print("  vector:", row.answer_vector[:300])

answers that differ: 22 of 22

 G003
  hybrid: Reviewers mention that the adapter doesn't fit their dryer [4] and that it won't hold their diffuser [5]. However, some reviewers, like [2] and [6], mention that it fits and works well with their diffuser. One reviewer, [1], mentions that it fits but gets loose with the heat of the dryer. [4][5][1][
  vector: Reviewers mention that the product doesn't always fit their blow dryers or diffusers, which may affect its value for money [3][5]. Some reviewers, however, report that it fits and works well, which suggests that the product can be a good value for those who find it compatible [2][6]. Overall, the va

 G004
  hybrid: Reviewer [4] states that it still doesn't fit their dryer, and reviewer [1] mentions that it gets loose with the heat of the dryer. Reviewers [2], [3], [5], and [6] all report that the adapter fits and works well with their respective products.
  vector: Reviewers say that the SnozzlePro Universal Hair Dryer Nozzle Adapter

## 5. Read 10 answers by hand

**What this does:** writes 10 random answered questions from the vector run into `07_hand_read.csv`, with the numbered review excerpts beside each answer. The automatic checks only see numbers and word overlap. **Only you reading can tell** if a sentence is really in the review.

For each row fill `verdict`:
- `ok`: every claim is in the review it cites
- `partly`: some claims are right, some are not in the cited review
- `wrong`: the main claim is not in the review, or it cites the wrong number

Write what went wrong in `note`. Then run the last cell of this notebook.

In [7]:
answered = results["vector"][~results["vector"]["refused"]]
sample = answered.sample(n=min(10, len(answered)), random_state=7)

hand = pd.DataFrame({
    "id": sample["id"],
    "question": sample["question"].str.slice(0, 160),
    "answer": sample["answer"],
    "excerpts": sample["passages"].apply(lambda text: "\n".join(f"[{i}] {p}" for i, p in enumerate(text.split(" || "), start=1))),
    "verdict": "",
    "note": "",
})
hand.to_csv(OUT / "07_hand_read.csv", index=False)
print("written:", OUT / "07_hand_read.csv", "| rows:", len(hand))

written: d:\Resume\Data Scientist\Resume_Projects\ShopSense_Product Review Intelligence Copilot\reports\eval\07_hand_read.csv | rows: 10


**Run this last cell only after you filled `verdict`.** It counts your verdicts.

In [8]:
filled = pd.read_csv(OUT / "07_hand_read.csv")
print(filled["verdict"].value_counts(dropna=False).to_string())

verdict
ok        8
partly    1
wrong     1
